## PART 1 - **Data Preprocessing**


### Dataset 1 : ***Load CMS Data***
1.	Load raw CMS datasets (2017–2023)
2. 	Merge into one dataframe
3.	Add Data_Year column

In [2]:
import pandas as pd
import os

# File paths 
#print(os.getcwd())
DATA_PATH = "../Data/"
#print(os.listdir(DATA_PATH))

Main_DataPath = DATA_PATH + "CMS_Data/"

df_2017 = pd.read_csv(Main_DataPath + "MUP_IHP_RY23_P03_V10_DY17_PRVSVC.csv.gz", encoding='latin-1')  # 2017 data
df_2018 = pd.read_csv(Main_DataPath + "MUP_IHP_RY23_P03_V10_DY18_PRVSVC.csv.gz", encoding='latin-1') # 2018 data
df_2019 = pd.read_csv(Main_DataPath + "MUP_IHP_RY23_P03_V10_DY19_PRVSVC.csv.gz", encoding='latin-1') # 2019 data
df_2020 = pd.read_csv(Main_DataPath + "MUP_IHP_RY23_P03_V10_DY20_PRVSVC.csv.gz", encoding='latin-1') # 2020 data
df_2021 = pd.read_csv(Main_DataPath + "MUP_IHP_RY23_P03_V10_DY21_PRVSVC.csv.gz", encoding='latin-1')  # 2021 data
df_2022 = pd.read_csv(Main_DataPath + "MUP_INP_RY24_P03_V10_DY22_PrvSvc.csv.gz", encoding='latin-1') # 2022 data
df_2023 = pd.read_csv(Main_DataPath + "MUP_INP_RY25_P03_V10_DY23_PrvSvc.csv.gz", encoding='latin-1') # 2023 data

# Add a column to indicate the year of the data
df_2017["Data_Year"] = 2017
df_2018["Data_Year"] = 2018
df_2019["Data_Year"] = 2019
df_2020["Data_Year"] = 2020
df_2021["Data_Year"] = 2021
df_2022["Data_Year"] = 2022
df_2023["Data_Year"] = 2023

# Combine all years into one DataFrame
df_all = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023], ignore_index=True)

# Display basic information about the combined DataFrame
print("Combined shape:", df_all.shape)

# Show the count of records per year
print(df_all["Data_Year"].value_counts().sort_index())



Combined shape: (1178407, 16)
Data_Year
2017    196086
2018    192069
2019    187719
2020    158375
2021    151989
2022    145742
2023    146427
Name: count, dtype: int64


### Dataset 2 : ***Load POS File - Hospital Characteristics***
### Integrate Hospital Characteristics and Ownership Information (In 3 Steps below)
This step enriches the Medicare inpatient dataset by incorporating hospital-level characteristics from the CMS Provider of Services (POS) file.
1. Load the POS dataset that contains hospital characteristics and bed capacity details.

2. Left Join the POS data to the Medicare inpatient dataset using the hospital identifier (Rndrng_Prvdr_CCN), keeping all inpatient records.

3. Map hospital ownership codes into ownership categories using the POS File data dictionary.

###  Step 1 - Load Hospital characteristics Dataset

In [3]:
# -----------------------------
# POS file is CSV (load only required columns)
# -----------------------------
POS_FILE = DATA_PATH + "HospitalType/POS_File_Hospital_Non_Hospital_Facilities_Q4_2023.csv.gz"
pos_cols = ["PRVDR_NUM", "FAC_NAME", "GNRL_CNTL_TYPE_CD", "CRTFD_BED_CNT", "BED_CNT"]

pos_df = pd.read_csv(
    POS_FILE,
    usecols=pos_cols,     # load ONLY these columns
   # dtype=str,            # keep as strings to preserve leading zeros
    encoding="latin-1"    
)

print("POS shape (selected cols):", pos_df.shape)
print(pos_df.head())

POS shape (selected cols): (114047, 5)
                                FAC_NAME PRVDR_NUM GNRL_CNTL_TYPE_CD  \
0        SOUTHEAST HEALTH MEDICAL CENTER     10001                 8   
1                 NORTH JACKSON HOSPITAL     10004                 8   
2  MARSHALL MEDICAL CENTERS SOUTH CAMPUS     10005                 8   
3           NORTH ALABAMA MEDICAL CENTER     10006                 4   
4               MIZELL MEMORIAL HOSPITAL     10007                 2   

   CRTFD_BED_CNT  BED_CNT  
0          420.0    420.0  
1           64.0     64.0  
2          240.0    240.0  
3          338.0    338.0  
4           99.0     99.0  


C:\Users\JYOTHI\AppData\Local\Temp\ipykernel_42168\3393140428.py:7: DtypeWarning: Columns (17,28) have mixed types. Specify dtype option on import or set low_memory=False.
  pos_df = pd.read_csv(


###  Step 2- Left join Hospital Characteristics

In [4]:
# JOIN: df_all with pos_df
# -----------------------------

# 1) Create temporary string keys (doesn't change other columns)
df_all["_ccn_key"] = (
    df_all["Rndrng_Prvdr_CCN"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

pos_df["_ccn_key"] = (
    pos_df["PRVDR_NUM"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# 2) Left join (keeps all Medicare rows)
df_medidata = df_all.merge(
    pos_df[["_ccn_key","FAC_NAME", "GNRL_CNTL_TYPE_CD", "CRTFD_BED_CNT", "BED_CNT"]],
    how="left",
    on="_ccn_key")


print("Joined shape:", df_medidata.shape)
print("Match rate (ownership not null):", df_medidata["GNRL_CNTL_TYPE_CD"].notna().mean()) # 100% = all records matched
print("Missing values in GNRL_CNTL_TYPE_CD:", df_medidata["GNRL_CNTL_TYPE_CD"].isna().sum()) # no of unmatched records

# 3) Clean up helper key columns
df_all.drop(columns=["_ccn_key"], inplace=True)
pos_df.drop(columns=["_ccn_key"], inplace=True)
df_medidata.drop(columns=["_ccn_key"], inplace=True)


# Preview joined columns
# Show rows 100-110 for the selected columns
df_medidata.iloc[100:111][["Rndrng_Prvdr_RUCA_Desc","Rndrng_Prvdr_CCN", "Rndrng_Prvdr_Org_Name", "FAC_NAME", "GNRL_CNTL_TYPE_CD", "CRTFD_BED_CNT", "BED_CNT"]]

Joined shape: (1178407, 21)
Match rate (ownership not null): 1.0
Missing values in GNRL_CNTL_TYPE_CD: 0


,Rndrng_Prvdr_RUCA_Desc,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,FAC_NAME,GNRL_CNTL_TYPE_CD,CRTFD_BED_CNT,BED_CNT
100,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
101,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
102,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
103,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
104,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
105,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
106,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
107,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
108,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0
109,Metropolitan area core: primary flow within an...,10001,Southeast Alabama Medical Center,SOUTHEAST HEALTH MEDICAL CENTER,8,420.0,420.0


### Step 3 - Map Hospital Ownership type

In [5]:
def map_ownership(code):
    if code.startswith(('1','2','3')) or code == '13':
        return 'For-Profit'
    elif code.startswith(('4','5','6')):
        return 'Non-Profit'
    elif code.startswith(('7','8','9','10','11','12')):
        return 'Government'
    else:
        return 'Unknown'
    
# Map ownership codes to descriptive categories
df_medidata['Ownership_Type'] = df_medidata['GNRL_CNTL_TYPE_CD'].astype(str).apply(map_ownership)


ownership_individual_map = {
    '1': 'Individual',
    '2': 'Partnership',
    '3': 'Corporation',
    '4': 'Church Related',
    '5': 'Corporation',
    '6': 'Other',
    '7': 'State',
    '8': 'County',
    '9': 'City',
    '10': 'City/County',
    '11': 'Hospital District',
    '12': 'Federal',
    '13': 'Limited Liability Corporation'
}

df_medidata['ownership_individual'] = (
    df_medidata['GNRL_CNTL_TYPE_CD']
    .astype(str)
    .str.extract(r'(\d+)')[0]   
    .map(ownership_individual_map)
)

    
df_medidata[['Ownership_Type','ownership_individual','FAC_NAME']].tail(5)



,Ownership_Type,ownership_individual,FAC_NAME
1178402,For-Profit,Partnership,METHODIST MIDLOTHIAN MEDICAL CENTER
1178403,For-Profit,Partnership,METHODIST MIDLOTHIAN MEDICAL CENTER
1178404,For-Profit,Partnership,METHODIST MIDLOTHIAN MEDICAL CENTER
1178405,Non-Profit,Church Related,TEXAS HEALTH HOSPITAL MANSFIELD
1178406,Non-Profit,Church Related,TEXAS HEALTH HOSPITAL MANSFIELD


## Check Null Values 

In [6]:
df_medidata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1178407 entries, 0 to 1178406
Data columns (total 22 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Rndrng_Prvdr_CCN           1178407 non-null  int64  
 1   Rndrng_Prvdr_Org_Name      1178407 non-null  object 
 2   Rndrng_Prvdr_City          1178407 non-null  object 
 3   Rndrng_Prvdr_St            1178407 non-null  object 
 4   Rndrng_Prvdr_State_FIPS    1178407 non-null  int64  
 5   Rndrng_Prvdr_Zip5          1178407 non-null  int64  
 6   Rndrng_Prvdr_State_Abrvtn  1178407 non-null  object 
 7   Rndrng_Prvdr_RUCA          1177690 non-null  float64
 8   Rndrng_Prvdr_RUCA_Desc     1177690 non-null  object 
 9   DRG_Cd                     1178407 non-null  int64  
 10  DRG_Desc                   1178407 non-null  object 
 11  Tot_Dschrgs                1178407 non-null  int64  
 12  Avg_Submtd_Cvrd_Chrg       1178407 non-null  float64
 13  Avg_Tot_Pymt

### Dataset 3 : ***Load DRG Severity Scores for Respective Years***
### LOAD AND JOIN DRG SEVERITY SCORES DATA FILES FOR RESPECTIVE YEARS

1. Load all DRG weight/severity Excel files from the Data/DRG_Scores/ folder (FY 2017–FY 2023).

2. Automatically detect the fiscal year (FY) for each file by scanning the top rows for “FY 20xx”.

3. Combine all yearly DRG tables into one consolidated dataframe (df_drg_all) for later joining.

In [7]:
import pandas as pd
import re
from pathlib import Path
import os


# ------------------------------------------------------------
# 1) Load all DRG score files (FY 2017–FY 2023) from folder
# ------------------------------------------------------------

DATA_PATH = "../Data/"

DRG_DIR = Path(DATA_PATH) / "DRG_Scores"   # uses your existing DATA_PATH = "../Data/"
drg_files = sorted(list(DRG_DIR.glob("*.xls*")))

print(f"Found {len(drg_files)} DRG files in: {DRG_DIR}")
# for f in drg_files:
#     print(" -", f.name)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
FY_PATTERN = re.compile(r"FY\s*(20\d{2})", re.IGNORECASE)

def detect_fy_from_top_rows(filepath: Path, scan_rows: int = 3) -> int:
    """
    Reads top rows (header=None) and finds 'FY 20xx' anywhere.
    """
    preview = pd.read_excel(filepath, header=None, nrows=scan_rows)
    text = " ".join(preview.fillna("").astype(str).values.flatten())
    m = FY_PATTERN.search(text)
    if not m:
        raise ValueError(f"Could not detect FY year in file: {filepath.name}")
    return int(m.group(1))

def detect_header_row(filepath: Path, scan_rows: int = 20) -> int:
    top = pd.read_excel(filepath, header=None, nrows=scan_rows)

    for i in range(len(top)):
        row = top.iloc[i].fillna("").astype(str).str.strip().str.lower()

        # Exact match prevents grabbing the title line ("... MS-DRGs ... weighting ...")
        has_msdrg = (row == "ms-drg").any()
        has_weights = (row == "weights").any() or (row == "weight").any()

        # Extra guard: real header usually also contains MDC / TYPE / title columns
        has_mdc = (row == "mdc").any()
        has_type = (row == "type").any()

        if has_msdrg and has_weights and (has_mdc or has_type):
            return i

    raise ValueError(f"Header row not found in first {scan_rows} rows: {filepath.name}")


def simple_load_table5(filepath: Path) -> pd.DataFrame:
    fy = detect_fy_from_top_rows(filepath, scan_rows=5)   # FY can be in first few rows
    header_row = detect_header_row(filepath, scan_rows=20)

    df = pd.read_excel(filepath, header=header_row)

    # Standardize column names for matching
    cols_lower = {c: str(c).strip().lower() for c in df.columns}

    ms_col = next((c for c, v in cols_lower.items() if v == "ms-drg" or "ms-drg" in v), None)
    wt_col = next((c for c, v in cols_lower.items() if v == "weights" or v == "weight" or "weight" in v), None)

    if ms_col is None or wt_col is None:
        raise ValueError(f"Could not find MS-DRG / Weights columns in: {filepath.name}")

    out = df[[ms_col, wt_col]].copy()
    out.columns = ["MS_DRG", "DRG_Weight"]
    out["FY"] = fy

    # Clean MS_DRG codes + make weights numeric
    out["MS_DRG"] = (
        out["MS_DRG"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(3)
    )
    out["DRG_Weight"] = pd.to_numeric(out["DRG_Weight"], errors="coerce")

    # Drop header-like or invalid rows (e.g., "MS-DRG" appearing as a data row)
    out = out[out["MS_DRG"].str.match(r"^\d{3}$", na=False)]
    out = out.dropna(subset=["DRG_Weight"])

    return out


#call the function to test
drg_tables = []

for f in drg_files:
    try:
        df_one_year = simple_load_table5(f)
        print(f"Loaded {f.name} | FY={df_one_year['FY'].iloc[0]} | Rows={len(df_one_year)}")
        drg_tables.append(df_one_year)
    except Exception as e:
        print(f"FAILED to load {f.name}: {e}")

print(f"Total DRG tables loaded: {len(drg_tables)}")

# combine all years into one DataFrame
df_drg_all = pd.concat(drg_tables, ignore_index=True)
print("Combined DRG shape:", df_drg_all.shape)
df_drg_all.head()



Found 7 DRG files in: ..\Data\DRG_Scores
Loaded CMS 1716 Table 5 FR and CN.xlsx | FY=2020 | Rows=759
Loaded CMS-1655-P FY 2017 TABLE 5 _Weight_File_FR.xlsx | FY=2017 | Rows=755
Loaded CMS-1677 Table 5_Weight_File_FR_CN.xlsx | FY=2018 | Rows=752
Loaded CMS-1694-F Table 5.xlsx | FY=2019 | Rows=759
Loaded CMS-1735-FR and CN Table 5.xlsx | FY=2021 | Rows=765
Loaded CMS-1752-F Table 5.xlsx | FY=2022 | Rows=765
Loaded CMS-1771-F Table 5.xlsx | FY=2023 | Rows=765
Total DRG tables loaded: 7
Combined DRG shape: (5320, 3)


,MS_DRG,DRG_Weight,FY
0,001,27.6339,2020
1,002,14.0137,2020
2,003,18.9539,2020
3,004,11.5438,2020
4,005,10.3127,2020


### Left Join DRG weights based on DGR_Cd and Year keys
Perform a left join to attach DRG weights to each Medicare record using the keys:

Data_Year + DRG_key (Medicare data)

FY + MS_DRG_key (DRG weights)

In [8]:
## JOIN dataframes
#  DRG code in medidata key
df_medidata["DRG_key"] = (
    df_medidata["DRG_Cd"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(3)
)

#drg code in drg_all key
df_drg_all["MS_DRG_key"] = (
    df_drg_all["MS_DRG"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(3)
)


# LEFT JOIN
df_medidata = df_medidata.merge(
    df_drg_all,
    how="left",
    left_on=["Data_Year", "DRG_key"],
    right_on=["FY", "MS_DRG_key"]
)

print(df_drg_all.columns.tolist())

print("Match count:", (df_medidata["DRG_key"] == df_medidata["MS_DRG_key"]).sum())


#Checks 
# % of rows where DRG weight was successfully attached
weight_match_rate = df_medidata["DRG_Weight"].notna().mean()
print("DRG weight match rate (not null):", weight_match_rate)

# Show a few rows that did NOT get a DRG weight
print("\nSample unmatched rows (no DRG weight):")
print(df_medidata.loc[df_medidata["DRG_Weight"].isna(),
                      ["Data_Year", "DRG_Cd", "DRG_key", "MS_DRG", "MS_DRG_key"]].head(10))


#Drop helper key columns
df_medidata.drop(columns=["DRG_key", "MS_DRG_key", "FY", "MS_DRG"], inplace=True)

['MS_DRG', 'DRG_Weight', 'FY', 'MS_DRG_key']
Match count: 1177503
DRG weight match rate (not null): 0.9992328626696888

Sample unmatched rows (no DRG weight):
        Data_Year  DRG_Cd DRG_key MS_DRG MS_DRG_key
501329       2019     319     319    NaN        NaN
575954       2020     522     522    NaN        NaN
576372       2020     522     522    NaN        NaN
576521       2020     522     522    NaN        NaN
576718       2020     650     650    NaN        NaN
576719       2020     651     651    NaN        NaN
577023       2020     522     522    NaN        NaN
577755       2020     522     522    NaN        NaN
577873       2020     522     522    NaN        NaN
577975       2020     522     522    NaN        NaN


## ***Final Merged Medicare Inpatient Dataset***

At this stage, all relevant data sources have been successfully combined into a single consolidated dataset (`df_medidata`). 

**The dataset includes Medicare inpatient utilization and payment information enriched with hospital characteristics, geographic attributes, and DRG severity weights across multiple years.**
This finalized dataset serves as the foundation for exploratory data analysis, utilization forecasting, and subsequent predictive modeling.


In [9]:
df_medidata.shape

(1178407, 23)

In [10]:
df_medidata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1178407 entries, 0 to 1178406
Data columns (total 23 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Rndrng_Prvdr_CCN           1178407 non-null  int64  
 1   Rndrng_Prvdr_Org_Name      1178407 non-null  object 
 2   Rndrng_Prvdr_City          1178407 non-null  object 
 3   Rndrng_Prvdr_St            1178407 non-null  object 
 4   Rndrng_Prvdr_State_FIPS    1178407 non-null  int64  
 5   Rndrng_Prvdr_Zip5          1178407 non-null  int64  
 6   Rndrng_Prvdr_State_Abrvtn  1178407 non-null  object 
 7   Rndrng_Prvdr_RUCA          1177690 non-null  float64
 8   Rndrng_Prvdr_RUCA_Desc     1177690 non-null  object 
 9   DRG_Cd                     1178407 non-null  int64  
 10  DRG_Desc                   1178407 non-null  object 
 11  Tot_Dschrgs                1178407 non-null  int64  
 12  Avg_Submtd_Cvrd_Chrg       1178407 non-null  float64
 13  Avg_Tot_Pymt

In [11]:
#List all column names in df
print("Column names:")
print(df_medidata.columns)


# Display the first and last 3 rows of the combined DataFrame
print("First 3 rows:")
print(df_medidata.head(3))
print("Last 3 rows:")
print(df_medidata.tail(3))



Column names:
Index(['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City',
       'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5',
       'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA',
       'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs',
       'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt',
       'Data_Year', 'FAC_NAME', 'GNRL_CNTL_TYPE_CD', 'CRTFD_BED_CNT',
       'BED_CNT', 'Ownership_Type', 'ownership_individual', 'DRG_Weight'],
      dtype='object')
First 3 rows:
   Rndrng_Prvdr_CCN             Rndrng_Prvdr_Org_Name Rndrng_Prvdr_City  \
0             10001  Southeast Alabama Medical Center            Dothan   
1             10001  Southeast Alabama Medical Center            Dothan   
2             10001  Southeast Alabama Medical Center            Dothan   

          Rndrng_Prvdr_St  Rndrng_Prvdr_State_FIPS  Rndrng_Prvdr_Zip5  \
0  1108 Ross Clark Circle                        1              36301   
1  1108 Ross

## PART 2 -  **Variable Type Classification**

Numerical variables include discharge counts, payment amounts, and bed capacity measures, which are used for statistical analysis and predictive modeling.

Categorical variables include hospital identifiers, geographic attributes, diagnosis-related group (DRG) codes, and rural–urban classifications, which are used for grouping and comparison analysis.

Text variables, such as hospital names and diagnosis descriptions, provide contextual information but are not directly used in numerical modeling.

In [12]:
numerical_cols = df_medidata.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df_medidata.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")


Numerical columns (13): ['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_RUCA', 'DRG_Cd', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt', 'Data_Year', 'CRTFD_BED_CNT', 'BED_CNT', 'DRG_Weight']

Categorical columns (10): ['Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Desc', 'FAC_NAME', 'GNRL_CNTL_TYPE_CD', 'Ownership_Type', 'ownership_individual']


In [13]:
#Qualitative summary -Categorical columns
df_medidata.describe(exclude='number') 

,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA_Desc,DRG_Desc,FAC_NAME,GNRL_CNTL_TYPE_CD,Ownership_Type,ownership_individual
count,1178407,1178407,1178407,1178407,1177690,1178407,1178407,1178407,1178407,1178407
unique,4073,1955,3549,51,15,622,3226,17,3,9
top,Methodist Hospital,New York,525 East 68th Street,CA,Metropolitan area core: primary flow within an...,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,METHODIST HOSPITAL,2,For-Profit,Partnership
freq,3094,10006,2626,96181,1008770,19303,3014,618169,856516,661935


In [14]:
# Quantitative summary - Numerical columns
df_medidata[['Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt','CRTFD_BED_CNT','BED_CNT']].describe()

,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt,CRTFD_BED_CNT,BED_CNT
count,1.178407e+06,1.178407e+06,1.178407e+06,1.178407e+06,1.178407e+06,1.178407e+06
mean,3.596944e+01,7.531559e+04,1.643991e+04,1.379085e+04,4.447039e+02,4.556215e+02
std,5.400602e+01,9.581047e+04,1.823228e+04,1.603071e+04,3.608699e+02,3.637111e+02
min,1.100000e+01,2.262917e+03,1.065091e+03,0.000000e+00,1.000000e+00,1.000000e+00
25%,1.400000e+01,2.897700e+04,7.569098e+03,5.970380e+03,2.150000e+02,2.210000e+02
50%,2.100000e+01,4.856575e+04,1.125342e+04,9.314641e+03,3.540000e+02,3.630000e+02
75%,3.700000e+01,8.659173e+04,1.816626e+04,1.520336e+04,5.710000e+02,5.920000e+02
max,4.291000e+03,1.041893e+07,7.617388e+05,7.514790e+05,3.251000e+03,3.251000e+03


In [15]:
df_medidata["GNRL_CNTL_TYPE_CD"] = (
    df_medidata["GNRL_CNTL_TYPE_CD"]
    .astype("string")
)

## PART 3 - **Missing data Analysis**

In [16]:
def analyze_missing_values(df):
    """
    Analyzes missing values in a dataframe.
    Returns a summary DataFrame showing missing counts and percentages by column.
    """
    missing_summary = pd.DataFrame({
        "Total_rows": df.shape[0],
        "Missing_Count": df.isna().sum(),
        "Missing_Percentage": (df.isna().mean() * 100).round(2)
    })
    missing_summary = (
        missing_summary
        .reset_index()
        .rename(columns={"index": "Column_Name"})
        .sort_values("Missing_Percentage", ascending=False)
    )
    return missing_summary

# Call the method with df_medidata
analyze_missing_values(df_medidata)

,Column_Name,Total_rows,Missing_Count,Missing_Percentage
22,DRG_Weight,1178407,904,0.08
7,Rndrng_Prvdr_RUCA,1178407,717,0.06
8,Rndrng_Prvdr_RUCA_Desc,1178407,717,0.06
13,Avg_Tot_Pymt_Amt,1178407,0,0.00
21,ownership_individual,1178407,0,0.00
20,Ownership_Type,1178407,0,0.00
19,BED_CNT,1178407,0,0.00
18,CRTFD_BED_CNT,1178407,0,0.00
17,GNRL_CNTL_TYPE_CD,1178407,0,0.00
16,FAC_NAME,1178407,0,0.00


Less than one percent (approximately 0.08%) of records contained missing values for rural–urban classification fields and DRG weights

#### RUCA Missing Data Analysis

In [17]:
# Replace missing or empty values in Rndrng_Prvdr_RUCA with 'Unknown'
df_medidata['Rndrng_Prvdr_RUCA'] = df_medidata['Rndrng_Prvdr_RUCA'].fillna(99)  # Use 99 for missing RUCA codes
df_medidata['Rndrng_Prvdr_RUCA_Desc'] = df_medidata['Rndrng_Prvdr_RUCA_Desc'].fillna('Unknown')

print("Missing values after replacement RUCA column:", df_medidata['Rndrng_Prvdr_RUCA'].isna().sum())
print("Missing values after replacement RUCA Desc column:", df_medidata['Rndrng_Prvdr_RUCA_Desc'].isna().sum())

Missing values after replacement RUCA column: 0
Missing values after replacement RUCA Desc column: 0


To preserve all inpatient records :

- Missing RUCA codes (`Rndrng_Prvdr_RUCA`) were replaced with a placeholder value (99) to  indicate an unknown classification.
- Missing RUCA descriptions(`Rndrng_Prvdr_RUCA_Desc`) were labeled as "Unknown".

To ensure no inpatient utilization data is lost and Geographic analyses remain inclusive
Given the minimal amount of missing data imputation using an explicit "Unknown" category was preferred over row removal.


### Analysis of Missing DRG Weights
Missing DRG weights were examined by year (`Data_Year`) and diagnosis code (`DRG_Cd`) to determine  systematic pattern. The analysis revealed that:
- Missing DRG weights were concentrated in a small number of DRG codes
- Most missing values occurred in specific year–DRG combinations
- The overall proportion of missing DRG weights was relatively small compared to the total dataset size

In [18]:
## Missing DRG weights Analysis year and code wise
def analyze_missing_drg_weights():
    """
    Analyzes missing DRG_Weight values in df_medidata.
    Returns a summary DataFrame showing missing DRG records by year.
    """
    missing = df_medidata[df_medidata["DRG_Weight"].isna()].copy()
    
    missing_drg_by_year = (
        missing.groupby(["Data_Year", "DRG_Cd"])
        .size()
        .reset_index(name="Count")
        .sort_values(["Count"], ascending=[False])
    )
    
    missing_drg_by_year["Count_Formatted"] = missing_drg_by_year["Count"].apply(lambda x: f"{x:,}")
    
    return missing_drg_by_year


# Call the method
analyze_missing_drg_weights()

,Data_Year,DRG_Cd,Count,Count_Formatted
10,2023,322,406,406
3,2020,522,259,259
9,2023,321,206,206
4,2020,650,13,13
2,2020,521,8,8
8,2023,277,3,3
1,2020,141,2,2
5,2020,651,2,2
11,2023,324,2,2
0,2019,319,1,1


### Imputation Strategy for Missing DRG Weights
**Step 1: DRG and Year -Specific Median Imputation**

- Missing DRG weights were imputed using the median value for the same DRG across available fiscal years

This approach maintains clinical relevance by ensuring that imputed values reflect the typical severity of the same diagnosis rather than using a generic replacement.

In [19]:
#df_drg_all.head()

# Ensure weight is numeric
df_drg_all["DRG_Weight"] = pd.to_numeric(df_drg_all["DRG_Weight"], errors="coerce")

# Median weight for each DRG across all years
med_y_drg = (df_drg_all.groupby(["MS_DRG_key"])["DRG_Weight"].median()).reset_index(name="Median_DRG_Weight")

#print(med_y_drg.head())

df_medidata["DRG_key"] = (
    df_medidata["DRG_Cd"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(3)
)

df_medidata = df_medidata.merge(
    med_y_drg,                 # columns: MS_DRG_key, Median_DRG_Weight
    how="left",
    left_on="DRG_key",
    right_on="MS_DRG_key"
)

# Fill ONLY missing DRG_Weight values
mask = df_medidata["DRG_Weight"].isna()
df_medidata.loc[mask, "DRG_Weight"] = df_medidata.loc[mask, "Median_DRG_Weight"]


#Drop helper columns
df_medidata.drop(
    columns=["MS_DRG_key", "Median_DRG_Weight", "DRG_key","Median_DRG_Weight_ByYear","DRG_Weight_Imputed"],
    inplace=True,
    errors="ignore"
)


df_medidata.columns





Index(['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City',
       'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5',
       'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA',
       'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs',
       'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt',
       'Data_Year', 'FAC_NAME', 'GNRL_CNTL_TYPE_CD', 'CRTFD_BED_CNT',
       'BED_CNT', 'Ownership_Type', 'ownership_individual', 'DRG_Weight'],
      dtype='object')

In [20]:
analyze_missing_drg_weights()

,Data_Year,DRG_Cd,Count,Count_Formatted
4,2023,322,406,406
3,2023,321,206,206
2,2023,277,3,3
5,2023,324,2,2
0,2020,999,1,1
1,2023,212,1,1


**Step 2: Overall Median Imputation for Unmatched DRGs**

A small number of DRG codes did not have any  weight information across the historical data. For these cases:

-The overall median DRG weight across the entire dataset was calculated and used to impute the remaining missing values

This strategy ensures that no records are dropped 

In [21]:
overall_median_drg_Weight = df_medidata["DRG_Weight"].median()
print("Median",overall_median_drg_Weight)

# Replace remaining missing DRG_Weight values with overall median
df_medidata["DRG_Weight"] = df_medidata["DRG_Weight"].fillna(overall_median_drg_Weight)

# Sanity check
print("Remaining missing DRG_Weight:", df_medidata["DRG_Weight"].isna().sum())



Median 1.3731
Remaining missing DRG_Weight: 0


After completing both imputation steps no missing values remained in the DRG_Weight column.
### Rationale for This Approach
- Preserves all inpatient records for analysis
- Prioritizes diagnosis-specific information where available
- Avoids unnecessary row deletion

### **FINAL MISSING DATA CHECK**

In [22]:
#Analyze all missing values
analyze_missing_values(df_medidata)


,Column_Name,Total_rows,Missing_Count,Missing_Percentage
0,Rndrng_Prvdr_CCN,1178407,0,0.0
12,Avg_Submtd_Cvrd_Chrg,1178407,0,0.0
21,ownership_individual,1178407,0,0.0
20,Ownership_Type,1178407,0,0.0
19,BED_CNT,1178407,0,0.0
18,CRTFD_BED_CNT,1178407,0,0.0
17,GNRL_CNTL_TYPE_CD,1178407,0,0.0
16,FAC_NAME,1178407,0,0.0
15,Data_Year,1178407,0,0.0
14,Avg_Mdcr_Pymt_Amt,1178407,0,0.0


## PART 4 - **DUPLICATES ANALYSIS**

Each row represents one unique hospital–DRG–year combination.

In [23]:
# Check duplicates for hospital–DRG–year records
duplicate_count = df_medidata.duplicated(
    subset=["Rndrng_Prvdr_CCN", "DRG_Cd", "Data_Year"]
).sum()

print("Duplicate hospital-DRG-year rows:", duplicate_count)


Duplicate hospital-DRG-year rows: 0


#### **Structural Checks** : Check if all numeric cols have numeric datatypes

In [24]:
# Verify numeric columns and their data types
numeric_cols = [
    "Tot_Dschrgs",
    "Avg_Submtd_Cvrd_Chrg",
    "Avg_Tot_Pymt_Amt",
    "Avg_Mdcr_Pymt_Amt",
    "CRTFD_BED_CNT",
    "BED_CNT",
    "DRG_Weight"
]

df_medidata[numeric_cols].dtypes


Tot_Dschrgs               int64
Avg_Submtd_Cvrd_Chrg    float64
Avg_Tot_Pymt_Amt        float64
Avg_Mdcr_Pymt_Amt       float64
CRTFD_BED_CNT           float64
BED_CNT                 float64
DRG_Weight              float64
dtype: object

***Drop Duplicate columns***

`Rndrng_Prvdr_Org_Name` and `FAC_Name` represent same data

In [25]:

df_medidata.drop('FAC_NAME', axis=1, inplace=True)
df_medidata.drop(['Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5'], axis=1, inplace=True)
df_medidata.columns

Index(['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City',
       'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA',
       'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs',
       'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt',
       'Data_Year', 'GNRL_CNTL_TYPE_CD', 'CRTFD_BED_CNT', 'BED_CNT',
       'Ownership_Type', 'ownership_individual', 'DRG_Weight'],
      dtype='object')

## SAVE DF 
### Save cleaned df_medidata to Processed/Data Folder

In [26]:
from pathlib import Path

# Because notebook is in Scripts/, go one level up to project root
BASE_DIR = Path("..")

# Target folder: Data/Processed_Data
PROCESSED_DIR = BASE_DIR / "Data" / "Processed_Data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)  # creates folder if it doesn't exist

# Save file
out_path = PROCESSED_DIR / "df_medidata.parquet"
df_medidata.to_parquet(out_path, index=False)

print("Saved df_medidata to:", out_path.resolve())


Saved df_medidata to: C:\Users\JYOTHI\OneDrive\ClassProjects\Medicare DataAnalysis\Data\Processed_Data\df_medidata.parquet
